In [1]:
import os, shutil, pathlib

root = pathlib.Path("/kaggle/working/bfcl")
root.mkdir(parents=True, exist_ok=True)
os.environ["BFCL_PROJECT_ROOT"] = str(root)

import bfcl_eval  # import AFTER setting the variable
pkg = pathlib.Path(bfcl_eval.__path__[0])
shutil.copy(pkg / ".env.example", root / ".env")   # config file BFCL reads from the root

print("root:", os.environ["BFCL_PROJECT_ROOT"])
print(".env exists:", (root / ".env").exists())

ModuleNotFoundError: No module named 'bfcl_eval'

In [ ]:
  !uv pip install --python /kaggle/working/bfclenv/bin/python soundfile
  !/kaggle/working/bfclenv/bin/bfcl --help | head -5

In [ ]:
import os, pathlib, shutil, subprocess

ROOT = pathlib.Path("/kaggle/working/bfcl_root"); ROOT.mkdir(parents=True, exist_ok=True)
VENV_PY = "/kaggle/working/bfclenv/bin/python"
BFCL    = "/kaggle/working/bfclenv/bin/bfcl"
PORT    = 1053

pkg_dir = subprocess.check_output(
    [VENV_PY, "-c", "import bfcl_eval; print(bfcl_eval.__path__[0])"], text=True).strip()
shutil.copy(f"{pkg_dir}/.env.example", ROOT / ".env")

BFCL_ENV = {**os.environ,
            "BFCL_PROJECT_ROOT": str(ROOT),
            "LOCAL_SERVER_ENDPOINT": "localhost",
            "LOCAL_SERVER_PORT": str(PORT)}

def run_bfcl(args):
    """Run the bfcl CLI in its own env and stream its output into this cell."""
    p = subprocess.Popen(args, env=BFCL_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[exit code {p.returncode}]")
    return p.returncode

run_bfcl([BFCL, "--help"])

In [ ]:
import time, uuid
from fastapi import FastAPI, Request
from starlette.concurrency import run_in_threadpool

def make_app(generate_fn, model_id):
    """generate_fn(prompt, temperature, max_tokens) -> (text, n_prompt_tokens, n_completion_tokens, finish_reason)"""
    app = FastAPI()

    @app.get("/v1/models")
    def list_models():
        return {"object": "list", "data": [{"id": model_id, "object": "model", "owned_by": "local"}]}

    @app.post("/v1/completions")
    async def completions(request: Request):
        body = await request.json()
        prompt = body["prompt"]
        if isinstance(prompt, list):
            prompt = prompt[0]
        text, n_in, n_out, finish = await run_in_threadpool(
            generate_fn, prompt, body.get("temperature", 0.0), body.get("max_tokens", 256))
        return {
            "id": "cmpl-" + uuid.uuid4().hex[:12],
            "object": "text_completion",
            "created": int(time.time()),
            "model": body.get("model", model_id),
            "choices": [{"text": text, "index": 0, "logprobs": None, "finish_reason": finish}],
            "usage": {"prompt_tokens": n_in, "completion_tokens": n_out, "total_tokens": n_in + n_out},
        }

    return app

In [ ]:
import gc, threading, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import uvicorn

MODEL_ID        = "Qwen/Qwen3-1.7B"
WEIGHTS         = MODEL_ID
NO_THINK_PREFIX = True
MAX_NEW_TOKENS_CAP = 768     # deviation: BFCL asks for up to 4096; tool calls never need that and HF generate is slow

# --- clean up anything from a previous run of this cell (server thread + GPU memory) ---
if "server" in globals():
    server.should_exit = True; time.sleep(3)
for _n in ("model", "tok"):
    if _n in globals():
        del globals()[_n]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(WEIGHTS)
model = AutoModelForCausalLM.from_pretrained(
    WEIGHTS, torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa").eval()
print(f"model on GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

GEN_LOCK = threading.Lock()
THINK_OFF = "<think>\n\n</think>\n\n"
ASSISTANT_TAG = "<|im_start|>assistant\n"

def generate_text(prompt, temperature, max_tokens):
    if NO_THINK_PREFIX and prompt.endswith(ASSISTANT_TAG):
        prompt = prompt + THINK_OFF
    enc = tok(prompt, return_tensors="pt").to(model.device)
    n_in = enc["input_ids"].shape[1]
    kw = dict(max_new_tokens=min(int(max_tokens), MAX_NEW_TOKENS_CAP), pad_token_id=tok.pad_token_id)
    if temperature and temperature > 0.01:
        kw.update(do_sample=True, temperature=float(temperature))
    else:
        kw.update(do_sample=False)       # BFCL's default temperature is 0.001 -> effectively greedy
    with GEN_LOCK, torch.no_grad():
        out = model.generate(**enc, **kw)
    new = out[0][n_in:]
    text = tok.decode(new, skip_special_tokens=True)
    finish = "length" if len(new) >= kw["max_new_tokens"] else "stop"
    return text, n_in, len(new), finish

app = make_app(generate_text, MODEL_ID)
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("server started on port", PORT)

In [ ]:
import requests, json
print(requests.get(f"http://127.0.0.1:{PORT}/v1/models").json())

fn = {"name": "get_weather", "description": "Get the current weather for a city.",
      "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}
prompt = ("<|im_start|>system\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\n"
          "You are provided with function signatures within <tools></tools> XML tags:\n<tools>\n" + json.dumps(fn) +
          "\n</tools>\n\nFor each function call, return a json object with function name and arguments within "
          "<tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n"
          "</tool_call><|im_end|>\n<|im_start|>user\nWhat's the weather in Pune?<|im_end|>\n<|im_start|>assistant\n")

r = requests.post(f"http://127.0.0.1:{PORT}/v1/completions",
                  json={"model": MODEL_ID, "prompt": prompt, "temperature": 0.001, "max_tokens": 256}).json()
print(repr(r["choices"][0]["text"]))
print(r["usage"])

In [ ]:
import random

CATS = ["simple_python", "multiple", "parallel", "parallel_multiple", "irrelevance",
        "live_simple", "live_multiple", "live_parallel", "live_parallel_multiple",
        "live_irrelevance", "live_relevance"]

SMOKE = True
N_PER_CAT = 2 if SMOKE else 25

rng = random.Random(0)
ids_by_cat = {}
for cat in CATS:
    path = pathlib.Path(pkg_dir) / "data" / f"BFCL_v4_{cat}.json"
    ids = [json.loads(l)["id"] for l in open(path) if l.strip()]
    ids_by_cat[cat] = rng.sample(ids, min(N_PER_CAT, len(ids)))

with open(ROOT / "test_case_ids_to_generate.json", "w") as f:
    json.dump(ids_by_cat, f, indent=1)
print({c: len(v) for c, v in ids_by_cat.items()}, "| total", sum(len(v) for v in ids_by_cat.values()))

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B-FC"          # BFCL's registry name for the handler (FC = native tool-calling format)
RUN_TAG    = "base_1p7b_nothink"
RESULT_DIR = ROOT / f"result_{RUN_TAG}"
SCORE_DIR  = ROOT / f"score_{RUN_TAG}"

t0 = time.time()
rc = run_bfcl([BFCL, "generate", "--model", MODEL_NAME, "--skip-server-setup", "--run-ids",
               "--result-dir", str(RESULT_DIR), "--num-threads", "1"])
print(f"generation took {time.time()-t0:.0f}s")

In [ ]:
import pandas as pd
from IPython.display import display

rc = run_bfcl([BFCL, "evaluate", "--model", MODEL_NAME, "--test-category", ",".join(CATS),
               "--partial-eval", "--result-dir", str(RESULT_DIR), "--score-dir", str(SCORE_DIR)])

for name in ["data_non_live.csv", "data_live.csv"]:
    p = SCORE_DIR / name
    if p.exists():
        print("\n==", name); display(pd.read_csv(p))
    else:
        print("missing", p)

In [ ]:
import glob

CAT_TO_LOOK_AT = "irrelevance"      # change me: simple_python, live_simple, live_irrelevance, ...
files = glob.glob(str(SCORE_DIR / "**" / f"*{CAT_TO_LOOK_AT}*score.json"), recursive=True)
print(files)
if files:
    rows = [json.loads(l) for l in open(files[0]) if l.strip()]
    print("summary:", rows[0])
    for r in rows[1:4]:
        print("\n" + "-" * 70)
        print("keys:", list(r.keys()))
        for k in ("id", "prompt", "model_result_raw", "model_result", "error", "error_type"):
            if k in r:
                print(f"{k}: {str(r[k])[:500]}")